# Structural Sources of Real-vs-Synthetic Separability

**Structural recovery test:** Do Graphical Lasso errors explain RF distinguishability?

**Purpose:** Test whether real-vs-synthetic separability is reduced when synthetic data are forced to recover the real conditional-dependence structure while preserving synthetic marginal distributions.

The main notebook establishes two observations: (1) a random forest can distinguish real from synthetic rows, and (2) Graphical Lasso finds conditional-dependence errors. This notebook tests whether those observations are functionally connected.

## Experimental logic

For one fixed real/synthetic dataset pair:

1. Fit real and synthetic Graphical Lasso models to the complete matrices.
2. Convert each fitted precision matrix to its implied dependence matrix.
3. Generate synthetic rows from a Gaussian copula whose dependence is moved from the synthetic structure toward the real structure by a controlled amount.
4. Map generated values through the original synthetic class-conditional empirical distributions, so feature marginals remain synthetic rather than being replaced with real values.
5. Recalculate RF-AUC across fresh repeated train/test splits using identical seeds for every condition.
6. Compare correctly aligned real structure with a feature-permuted real-structure control.

**Hypothesis:** If structurally incorrect dependencies contribute to distinguishability, then increasing alignment with the correctly matched real structure should reduce RF-AUC more than alignment with a feature-permuted control structure.

**Evidence against the hypothesis:** structural error falls but AUC does not, or the permuted-structure control produces an equivalent AUC reduction.

In [ ]:
from pathlib import Path
import importlib
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
pkg_root = repo_root / "data_synthesis"
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

from src.revision import cache as revision_cache
from src.revision import config as revision_config
from src.revision import data_io
from src.revision import figure2_auc
from src.revision import figure4_graphical_lasso
from src.revision.common import class_counts
from src.revision.figure4_graphical_lasso import FIGURE4_ALPHAS
from src.revision import structural_repair

# Reload local helper edits when this cell is rerun in an existing Jupyter kernel.
structural_repair = importlib.reload(structural_repair)
ERROR_CATEGORIES = structural_repair.ERROR_CATEGORIES
build_edge_discrepancy_table = structural_repair.build_edge_discrepancy_table
plot_edge_discrepancy_map = structural_repair.plot_edge_discrepancy_map
plot_figure4_compatible_edge_status = structural_repair.plot_figure4_compatible_edge_status
plot_repair_dose_response = structural_repair.plot_repair_dose_response
run_structural_recovery_test = structural_repair.run_structural_recovery_test
plot_structural_recovery = structural_repair.plot_structural_recovery
summarize_structural_recovery = structural_repair.summarize_structural_recovery


ImportError: cannot import name 'plot_figure4_compatible_edge_status' from 'src.revision.structural_repair' (c:\Users\tonyt\Desktop\distinguishable_data\data_synthesis\src\revision\structural_repair.py)

In [ ]:
# Start with one dataset-generator pair so every diagnostic remains inspectable.
DATASET = "HIV"
METHOD = "CVAE"
ALIGNMENT_STRENGTHS = (0.0, 0.25, 0.5, 0.75, 1.0)
COPULA_REPEATS = 5
AUC_REPEATS = 10
FORCE_RECOMPUTE = False

CACHE_VERSION = 3
cache_name = f"structural_recovery_v{CACHE_VERSION}_{DATASET}_{METHOD}".lower().replace(" ", "_") + ".pkl"
experiment_cache = revision_config.CACHE_DIR / cache_name
experiment_cache


## Reference 1: RF origin separability

This is the same cached RF-AUC measurement used by `make_main_figures_revision.ipynb`. It defines the functional outcome that the repair experiment tries to reduce.

In [ ]:
auc_runs = revision_cache.get_auc_runs(force=False)
selected_auc = auc_runs[(auc_runs["dataset"] == DATASET) & (auc_runs["method"] == METHOD)]
display(selected_auc["separability_auc"].describe().to_frame("RF origin AUC"))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
figure2_auc._plot_auc_violin_panel(ax, auc_runs, DATASET, "A")
ax.set_title(f"{DATASET}: baseline real-vs-synthetic separability", loc="left", weight="bold")
plt.show()


In [ ]:
datasets, dataset_summary = data_io.initialize_datasets()
data = datasets[DATASET]
y_real = np.asarray(data["y"], dtype=int)

# Use the exact shared Figure 4 generation path from the main notebook.
figure4_real, figure4_synthetic, figure4_feature_names = figure4_graphical_lasso._get_figure4_precision_inputs(
    seed=revision_config.SEED,
    cvae_epochs=revision_config.CVAE_EPOCHS,
)
X_real = np.asarray(figure4_real[DATASET], dtype=np.float64)
X_syn = np.asarray(figure4_synthetic[DATASET][METHOD], dtype=np.float64)
feature_names = list(figure4_feature_names[DATASET])
n0, n1 = class_counts(y_real)
y_syn = np.r_[np.zeros(n0, dtype=int), np.ones(n1, dtype=int)]

display(dataset_summary)
print(f"Selected comparison: {DATASET} / {METHOD}; real={X_real.shape}, synthetic={X_syn.shape}")


## Reference 2: Graphical Lasso relationship errors

The first panel below is deliberately constrained to be apples-to-apples with the main notebook: it uses the same synthetic matrix, Graphical Lasso alpha, hierarchical feature order, edge threshold, four status categories, colors, and top-left matrix origin. Display index 1 means the first feature in the shared clustered order.

The second panel keeps that exact feature order but subdivides edges present in both graphs into `changed` and `reversed` categories. These categories describe the structural mismatch; the recovery intervention below uses the complete precision-implied dependence matrix rather than selecting individual displayed clusters or edges.

In [ ]:
fig_reference, figure4_order = plot_figure4_compatible_edge_status(
    X_real,
    X_syn,
    feature_names,
    alpha=FIGURE4_ALPHAS[DATASET],
    method=METHOD,
)
plt.show()

edge_table_full = build_edge_discrepancy_table(
    X_real,
    X_syn,
    feature_names=feature_names,
    alpha=FIGURE4_ALPHAS[DATASET],
    changed_threshold=0.05,
)

display(edge_table_full["category"].value_counts().rename_axis("edge status").to_frame("count"))
display(edge_table_full[edge_table_full["category"].isin(ERROR_CATEGORIES)].head(20))
plot_edge_discrepancy_map(
    edge_table_full,
    feature_names,
    title=f"{DATASET} / {METHOD}: repair-candidate detail in the same feature order",
    order=figure4_order,
)
plt.show()


## Controlled structural recovery experiment

The intervention does not edit or rearrange the original CVAE rows. It generates a new comparison sample from CVAE's class-conditional empirical feature distributions while controlling the Gaussian-copula dependence matrix. At alignment strength 0, dependence comes from the synthetic Graphical Lasso fit. At alignment strength 1, dependence comes from the correctly feature-matched real Graphical Lasso fit. Intermediate values interpolate between them.

The feature-permuted control uses the same real dependence matrix with feature identities shuffled. This controls for imposing equally strong structure without imposing the correct feature relationships.

In [ ]:
if experiment_cache.exists() and not FORCE_RECOMPUTE:
    with experiment_cache.open("rb") as handle:
        cached = pickle.load(handle)
    recovery_results = cached["recovery_results"]
    structures = cached["structures"]
    print(f"Loaded {experiment_cache.name}")
else:
    recovery_results, structures = run_structural_recovery_test(
        dataset_name=DATASET,
        method=METHOD,
        X_real=X_real,
        y_real=y_real,
        X_syn=X_syn,
        y_syn=y_syn,
        strengths=ALIGNMENT_STRENGTHS,
        copula_repeats=COPULA_REPEATS,
        auc_repeats=AUC_REPEATS,
        seed=revision_config.SEED,
    )
    with experiment_cache.open("wb") as handle:
        pickle.dump(
            {"recovery_results": recovery_results, "structures": structures},
            handle,
        )
    print(f"Wrote {experiment_cache.name}")


In [ ]:
fig, recovery_summary = plot_structural_recovery(
    recovery_results,
    title=f"{DATASET} / {METHOD}: conditional-structure recovery test",
)
plt.show()
display(recovery_summary)


## Control and validity checks

Because the adjusted rows are newly sampled, finite-sample marginal distributions will not be numerically identical to the original CVAE columns. They are sampled from the same class-conditional empirical distributions. Kolmogorov-Smirnov statistics quantify the remaining sampling variation.

A convincing result requires three linked patterns: correctly aligned structure approaches the real structure, its AUC falls relative to the strength-0 copula baseline, and the feature-permuted control does not show the same AUC reduction.

In [ ]:
validity_summary = recovery_summary[
    [
        "condition",
        "strength",
        "mean_structural_error",
        "mean_marginal_ks",
        "max_marginal_ks",
        "mean_auc",
        "mean_auc_reduction",
    ]
]
display(validity_summary)

full_alignment = recovery_summary[np.isclose(recovery_summary["strength"], 1.0)]
display(full_alignment.sort_values("condition"))


## What structure is being imposed?

The intervention uses the complete real precision-implied dependence matrix, not visually defined clusters and not a hand-selected list of edges. Feature ordering therefore has no role in the intervention. The permuted control retains the same matrix values and eigenstructure but assigns them to the wrong feature identities.

In [ ]:
display(
    pd.DataFrame(
        {
            "original_feature_index": np.arange(1, len(feature_names) + 1),
            "feature_name": feature_names,
            "permuted_control_source_index": structures["permutation"] + 1,
        }
    )
)


## Decision rule

The hypothesis is supported for this dataset-generator pair only if:

- correctly aligned structure moves progressively closer to the real structure;
- RF-AUC decreases progressively from the strength-0 copula baseline;
- the AUC reduction exceeds the feature-permuted structural control; and
- synthetic class-conditional marginals remain close across alignment strengths.

If structural error decreases without a targeted AUC reduction, the correct conclusion is that the Graphical Lasso errors do not explain RF separability under this intervention. The remaining discriminator signal may instead be marginal, nonlinear, higher-order, or outside the Gaussian-copula representation.